In [3]:
#Scraper for Untappd
import time
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def beer_data(i,beer):
    #print(beer,i)
    #bid=beer attributes to be saved into CSV!
    beer_bid = beer.get_attribute("data-bid")
    beer_link = beer.find_element(By.CSS_SELECTOR, "p.name a").get_attribute("href")
    beer_abv = beer.find_element(By.CSS_SELECTOR, "p.abv").text
    beer_ibu = beer.find_element(By.CLASS_NAME, "ibu").text
    beer_rating = beer.find_element(By.CLASS_NAME, "rating-container").text
    beer_raters = beer.find_element(By.CLASS_NAME, "raters").text 
    beer_date = beer.find_element(By.CLASS_NAME, "date").text
    #print("\nbid:",beer_bid,"beer link:",beer_link,"abv:",beer_abv,"ibu:", beer_ibu,"rating:", beer_rating, "nr of raters:",beer_raters, "adding date:", beer_date)

    #data QA section, conversions, etc.
    #ABV
    if (beer_abv != "N/A") and (beer_abv):
        beer_abv = beer_abv.split("%")[0]
        beer_abv = float(beer_abv)
    else:
        beer_abv = "N/A"
    #IBU
    if (beer_ibu != "N/A IBU") and (beer_ibu):
        beer_ibu = beer_ibu.split(" ")[0]
        beer_ibu = int(beer_ibu)
    else:
        beer_ibu = "N/A"
    #Rating
    if beer_rating:
        beer_rating = beer_rating[1:-1]
        beer_rating = float(beer_rating)
    else:
        beer_rating = "N/A"
    #Raters
    if beer_raters:
        beer_raters = beer_raters.split(" ")[0]
        beer_raters = beer_raters.replace(",","")
        beer_raters = int(beer_raters)
    else:
        beer_raters = "N/A"
    #Adding date
    if beer_date:
        beer_date = beer_date.split(" ")[1]
    else:
        beer_date = "N/A"
    
    print("nr of entry",i,"beer id:",beer_bid,"beer link:",beer_link,"abv:",beer_abv,"ibu:", beer_ibu,"rating:", beer_rating, "nr of raters:",beer_raters, "adding date:", beer_date)
    #step back to main page to iterate next beer
    ##driver.back()
    time.sleep(2)

#ChromeDriver folder
#CHROMEDRIVER_PATH = r"C:\Program Files\Google\Chrome\Application\chromedriver-win64\chromedriver.exe"

#URL
url = "https://untappd.com/beer/top_rated?country=hungary"
options = Options()
# options.add_argument("--headless=new") with headless it does not work somehow
driver = webdriver.Chrome()

#waiting 15s until the page loads
try:
    #open url
    driver.get(url)

    #waiting 15s until the page loads
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    print("Url loaded successfully", driver.title)
    
    wait = WebDriverWait(driver, 1)
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    print(f"Iframe-ek száma: {len(iframes)}")
    clicked = False

    for i, iframe in enumerate(iframes):
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(iframe)

            elfogad = wait.until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[normalize-space()='Elfogad' or @aria-label='Elfogad']"
                ))
            )

            elfogad.click()
            print(f"✅ Elfogad gomb kattintva (iframe #{i})")
            clicked = True
            break

        except (TimeoutException, NoSuchElementException):
            print(f"❌ Nincs Elfogad gomb az iframe #{i}-ben")

    driver.switch_to.default_content()
    if not clicked:
        print("⚠️ Nem találtam Elfogad gombot egyik iframe-ben sem. Futtasd újra!")
        quit()
        
    #here starts the scraping
    driver.switch_to.default_content()           
    print("\nEnnyi sör van:\n")
    time.sleep(1)
    beers = driver.find_elements(By.CLASS_NAME,'beer-item')
    #beers = driver.find_elements(By.CSS_SELECTOR, ".beer-item")
    print(len(beers))

    for i in range (0,(len(beers))):
    #for i in range (0,2):
        time.sleep(3)
        beer=beers[i]
        beer_data(i,beer)
        
    #children = parent.find_elements(By.XPATH, "./div")
    #print(len(children))
    #time.sleep(15)

finally:
    #closing driver
    driver.quit()
    print("\nDriver Closed")

Url loaded successfully Top Rated Beers from Hungary – Top Rated Beers - Untappd
Iframe-ek száma: 8
❌ Nincs Elfogad gomb az iframe #0-ben
❌ Nincs Elfogad gomb az iframe #1-ben
❌ Nincs Elfogad gomb az iframe #2-ben
❌ Nincs Elfogad gomb az iframe #3-ben
❌ Nincs Elfogad gomb az iframe #4-ben
❌ Nincs Elfogad gomb az iframe #5-ben
❌ Nincs Elfogad gomb az iframe #6-ben
✅ Elfogad gomb kattintva (iframe #7)

Ennyi sör van:

50
nr of entry 0 beer id: 5826629 beer link: https://untappd.com/b/b-bop-fermentory-uuuuuuuu/5826629 abv: 4.5 ibu: N/A rating: 4.23 nr of raters: 944 adding date: 05/13/24
nr of entry 1 beer id: 6030210 beer link: https://untappd.com/b/horizont-brewing-night-shift-vintage-2024-imperial-pastry-stout-aged-in-bourbon-barrels-with-hazelnut-and-caramel/6030210 abv: 11.8 ibu: N/A rating: 4.22 nr of raters: 1403 adding date: 10/22/24

Driver Closed
